In [16]:
import os

def load_emails_from_folder(folder_path, label):
    emails = []
    labels = []
    
    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)
        
        if os.path.isfile(filepath):
            with open(filepath, "r", encoding="latin1") as f:
                emails.append(f.read())
                labels.append(label)
    
    return emails, labels

In [17]:
base_path = "data"

ham_folders = ["easy_ham", "easy_ham_2", "hard_ham"]
spam_folders = ["spam", "spam_2"]

emails = []
labels = []

# ham = 0
for folder in ham_folders:
    e, l = load_emails_from_folder(os.path.join(base_path, folder), 0)
    emails.extend(e)
    labels.extend(l)

# spam = 1
for folder in spam_folders:
    e, l = load_emails_from_folder(os.path.join(base_path, folder), 1)
    emails.extend(e)
    labels.extend(l)

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    emails,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels  # IMPORTANT
)

In [19]:
X_train[0]

'From jennifer3530t76@usa.com  Mon Aug 26 15:14:44 2002\nReturn-Path: <jennifer3530t76@usa.com>\nDelivered-To: zzzz@localhost.example.com\nReceived: from localhost (localhost [127.0.0.1])\n\tby phobos.labs.example.com (Postfix) with ESMTP id 1439B44169\n\tfor <zzzz@localhost>; Mon, 26 Aug 2002 10:13:02 -0400 (EDT)\nReceived: from mail.webnote.net [193.120.211.219]\n\tby localhost with POP3 (fetchmail-5.9.0)\n\tfor zzzz@localhost (single-drop); Mon, 26 Aug 2002 15:13:02 +0100 (IST)\nReceived: from smtp.easydns.com (smtp.easydns.com [205.210.42.30])\n\tby webnote.net (8.9.3/8.9.3) with ESMTP id QAA21963\n\tfor <zzzz@example.com>; Sun, 25 Aug 2002 16:29:10 +0100\nFrom: jennifer3530t76@usa.com\nReceived: from usa.com (unknown [61.157.85.117])\n\tby smtp.easydns.com (Postfix) with SMTP id 3606A2FDDF\n\tfor <zzzz@example.com>; Sun, 25 Aug 2002 11:28:39 -0400 (EDT)\nReceived: from unknown (HELO mx.loxsystems.net) (114.106.224.38)\n\tby rly-xl05.dohuya.com with NNFMP; Sun, 25 Aug 0102 02:03:20

In [20]:
import re

def preprocess_email(text, remove_headers=True):
    
    # supprimer headers
    if remove_headers:
        text = text.split("\n\n", 1)[-1]
    
    # minuscule
    text = text.lower()
    
    # remplacer URLs
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    
    # remplacer nombres
    text = re.sub(r"\d+", " NUMBER ", text)
    
    # supprimer ponctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # espaces multiples
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [21]:
from sklearn.base import BaseEstimator, TransformerMixin

class EmailPreprocessor(BaseEstimator, TransformerMixin):
    
    def __init__(self, remove_headers=True):
        self.remove_headers = remove_headers
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return [preprocess_email(email, self.remove_headers) for email in X]

In [22]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocess", EmailPreprocessor()),
    ("vectorizer", CountVectorizer(binary=True))  # présence (0/1)
])

In [23]:
CountVectorizer(
    binary=True,              # présence ou fréquence
    stop_words="english",     # supprimer mots inutiles
    max_features=5000,        # limiter vocabulaire
    ngram_range=(1,2)         # unigram + bigram
)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [24]:
pipeline = Pipeline([
    ("preprocess", EmailPreprocessor()),
    ("vectorizer", CountVectorizer(
        binary=True,
        stop_words="english",
        max_features=10000,
        ngram_range=(1,2)
    ))
])

In [25]:
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        ngram_range=(1,2)
    )),
    ("clf", LogisticRegression(max_iter=1000))
])

In [30]:
from sklearn.naive_bayes import MultinomialNB

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

In [31]:
from sklearn.metrics import classification_report

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1391
           1       0.96      0.94      0.95       480

    accuracy                           0.98      1871
   macro avg       0.97      0.97      0.97      1871
weighted avg       0.98      0.98      0.98      1871



In [32]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "tfidf__max_features": [5000, 10000],
    "tfidf__ngram_range": [(1,1), (1,2)],
    "clf__C": [0.1, 1, 10]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print(grid.best_params_)

{'clf__C': 10, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}


In [33]:
y_scores = pipeline.predict_proba(X_test)[:,1]

# exemple seuil personnalisé
y_pred_custom = (y_scores > 0.7).astype(int)

In [34]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_scores)